# Thử nghiệm cơ bản

In [ ]:
import pandas as pd

# Cách 1: Sử dụng encoding latin1 (thường thành công với các file Excel/CSV cũ)
df = pd.read_csv('test.csv', encoding='latin1')

# Cách 2: Ép kiểu utf-8 nhưng bỏ qua ký tự lỗi
# df = pd.read_csv('test.csv', encoding='utf-8', errors='replace')
# Chuyển đổi sang kiểu datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True)
# Ép kiểu về số, nếu có lỗi (ví dụ dính chữ) sẽ chuyển thành NaN
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')

# Xóa các dòng có Sales bị trống hoặc âm (nếu có)
df = df[df['Sales'] > 0]
categories = df['Category'].unique()
# Sau đó bạn có thể dùng danh sách này để INSERT vào bảng Categories trong SQL
# Tạm thời gán số lượng là 1 để không bị lỗi NOT NULL trong SQL
df['Quantity'] = 1
# 1. Xóa các dòng bị trống các thông tin bắt buộc
print(f"Số dòng TRƯỚC khi xóa missing: {len(df)}")
df = df.dropna(subset=['Order Date', 'Product Name', 'Sales'])
print(f"Số dòng SAU khi xóa missing: {len(df)}")

# 2. Xóa các dòng bị trùng lặp hoàn toàn
print(f"\n=== Thống kê trùng lặp ===")
print(f"Số dòng TRƯỚC khi xóa duplicates: {len(df)}")

# Đếm số dòng bị trùng lặp
num_duplicates = df.duplicated().sum()
print(f"Số dòng trùng lặp: {num_duplicates}")

df = df.drop_duplicates()
print(f"Số dòng SAU khi xóa duplicates: {len(df)}")
print(f"Tổng dòng bị loại bỏ: {num_duplicates}")
categories_clean = df[['Category']].drop_duplicates().rename(columns={'Category': 'category_name'})
# Lấy Product Name và Category tương ứng để biết sản phẩm thuộc nhóm nào
products_clean = df[['Product Name', 'Category']].drop_duplicates()

# Trích xuất dữ liệu Sales cơ bản

In [ ]:
import pandas as pd

# 1. Đọc file (Dùng latin1 để tránh lỗi Unicode byte 0xef trước đó)
df = pd.read_csv('test.csv', encoding='latin1')

# 2. TẠO CỘT quantity TRƯỚC (Vì file gốc không có)
# Chúng ta giả định mỗi dòng là 1 sản phẩm bán ra
df['quantity'] = 1 

# 3. Chuyển đổi ngày tháng để tránh lỗi định dạng khi vào SQL
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)

# 4. Bây giờ mới thực hiện lấy các cột để làm sạch (Lúc này 'quantity' đã tồn tại)
sales_clean = df[['Product Name', 'quantity', 'Order Date', 'Sales']]

# 5. Đổi tên cột cho giống với bảng Sales trong SQL của bạn (tùy chọn)
sales_clean.columns = ['product_name', 'quantity', 'sale_date', 'revenue']

print("Đã trích xuất dữ liệu thành công!")
print(sales_clean.head())

# 1. Làm sạch dữ liệu, xử lý Outlier và Feature Engineering (Cải tiến Outlier theo từng nhóm sản phẩm)

In [ ]:
import pandas as pd
import sys

try:
    # Đọc file với encoding an toàn nhất
    df = pd.read_csv('test.csv', encoding='latin1')
    
    # 1. Làm sạch ngày tháng
    print("\n" + "="*70)
    print("BƯỚC 1: CHUYỂN ĐỔI NGÀY THÁNG")
    print("="*70)
    print("\n--- Dữ liệu cột 'Order Date' TRƯỚC khi chuyển đổi ---")
    print(df['Order Date'].head())
    sys.stdout.flush()

    df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)

    print("\n--- Dữ liệu cột 'Order Date' SAU khi chuyển đổi ---")
    print(df['Order Date'].head())
    sys.stdout.flush()

    # 2. Làm sạch số liệu
    print("\n" + "="*70)
    print("BƯỚC 2: CHUYỂN ĐỔI KIỂU DỮ LIỆU")
    print("="*70)
    df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')
    df['quantity'] = 1

    # 3. Loại bỏ hàng lỗi/trống
    print("\n" + "="*70)
    print("BƯỚC 3: XỬ LÝ DỮ LIỆU THIẾU (MISSING VALUES)")
    print("="*70)
    print("\n--- Thống kê TRƯỚC khi xử lý dữ liệu thiếu ---")
    print(df.isnull().sum())
    print(f"\nSố dòng trước khi dropna: {len(df)}")
    sys.stdout.flush()

    df = df.dropna(subset=['Order Date', 'Product Name', 'Sales'])

    print("\n--- Thống kê SAU khi xử lý dữ liệu thiếu ---")
    print(df.isnull().sum())
    print(f"\nSố dòng sau khi dropna: {len(df)}")
    sys.stdout.flush()

    # ----- 4. Xử lý outlier -----
    print("\n" + "="*70)
    print("BƯỚC 4: XỬ LÝ OUTLIER (TÍNH RIÊNG THEO TỪNG SẢN PHẨM)")
    print("="*70)
    rows_before = len(df)
    print(f"\nSố dòng TRƯỚC xử lý outlier: {rows_before}")
    sys.stdout.flush()
    
    # Z-score theo nhóm sản phẩm
    print("\n" + "-"*70)
    print("--- Z-Score Method (Chỉ tính cho dữ liệu của cùng 1 sản phẩm) ---")
    print("-"*70)
    # Tinh Z-score dựa trên mean và std của TỪNG SẢN PHẨM riêng lẻ thay vì toàn cục
    df['Sales_zscore'] = df.groupby('Product Name')['Sales'].transform(lambda x: ((x - x.mean()) / x.std()).abs())
    # Các sản phẩm chỉ có 1 đơn hàng sẽ có std=0, nên z-score bị NaN, ta giữ lại (điền = 0)
    df['Sales_zscore'] = df['Sales_zscore'].fillna(0)
    
    rows_before_zscore = len(df)
    zscore_outliers = (df['Sales_zscore'] >= 3).sum()
    
    print(f"Số dòng TRƯỚC Z-Score: {rows_before_zscore}")
    print(f"Số ngoại lai phát hiện (|z| >= 3): {zscore_outliers}")
    print(f"Ngưỡng loại bỏ: |z-score| >= 3")
    sys.stdout.flush()
    
    df = df[df['Sales_zscore'] < 3]
    rows_after_zscore = len(df)
    print(f"Số dòng SAU Z-Score: {rows_after_zscore}")
    print(f"Dòng bị loại bỏ: {rows_before_zscore - rows_after_zscore}")
    sys.stdout.flush()

    # IQR theo nhóm sản phẩm
    print("\n" + "-"*70)
    print("--- IQR Method (Tính riêng Q1, Q3 cho từng sản phẩm) ---")
    print("-"*70)
    Q1 = df.groupby('Product Name')['Sales'].transform(lambda x: x.quantile(0.25))
    Q3 = df.groupby('Product Name')['Sales'].transform(lambda x: x.quantile(0.75))
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    rows_before_iqr = len(df)
    outlier_mask = (df['Sales'] < lower_bound) | (df['Sales'] > upper_bound)
    iqr_outliers = outlier_mask.sum()
    
    print(f"Số dòng TRƯỚC IQR: {rows_before_iqr}")
    print(f"Số ngoại lai phát hiện: {iqr_outliers}")
    sys.stdout.flush()
    
    df = df[~outlier_mask]
    rows_after_iqr = len(df)
    print(f"Số dòng SAU IQR: {rows_after_iqr}")
    print(f"Dòng bị loại bỏ: {rows_before_iqr - rows_after_iqr}")
    sys.stdout.flush()
    
    print("\n" + "="*70)
    print(f"TÓM TẮT: Loại bỏ {rows_before - rows_after_iqr} dòng ngoại lai ({(rows_before - rows_after_iqr)/rows_before*100:.2f}%)")
    print(f"Dòng ban đầu: {rows_before} → Dòng cuối: {rows_after_iqr}")
    print("="*70 + "\n")
    sys.stdout.flush()
    
    df = df.drop(columns=['Sales_zscore'])

    # ----- 5. Feature engineering cơ bản -----
    # Sort theo sản phẩm và ngày để tạo lag / rolling đúng
    df = df.sort_values(by=['Product Name', 'Order Date'])

    # Nếu cần nhóm theo product để tạo lag trong series riêng
    df['lag_1_sales'] = df.groupby('Product Name')['Sales'].shift(1)
    df['lag_2_sales'] = df.groupby('Product Name')['Sales'].shift(2)

    # Rolling mean (trên tổng doanh số theo ngày sắp xếp, theo product)
    df['rolling_7d_mean'] = df.groupby('Product Name')['Sales'].transform(lambda s: s.rolling(window=7, min_periods=1).mean())
    df['rolling_30d_mean'] = df.groupby('Product Name')['Sales'].transform(lambda s: s.rolling(window=30, min_periods=1).mean())

    # Xuất ra file sạch để bạn kiểm tra
    df.to_csv('cleaned_data.csv', index=False, encoding='utf-8-sig')
    print("✓ Đã làm sạch và lưu vào file 'cleaned_data.csv'")

except Exception as e:
    print(f"Vẫn còn lỗi: {e}")

# 2. Tách dữ liệu thành các bảng (Categories, Products, Sales)

In [ ]:
import pandas as pd

# Đọc file đã làm sạch
df = pd.read_csv('cleaned_data.csv', encoding='utf-8-sig')

# 1. Tạo file cho bảng Categories (Lấy danh mục duy nhất)
categories = df[['Category']].drop_duplicates().reset_index(drop=True)
categories.columns = ['category_name']
categories.to_csv('data_categories.csv', index=False, encoding='utf-8-sig')

# 2. Tạo file cho bảng Products
products = df[['Product Name', 'Category']].drop_duplicates().reset_index(drop=True)
products.columns = ['product_name', 'category_name']
products.to_csv('data_products.csv', index=False, encoding='utf-8-sig')

# 3. Tạo file cho bảng Sales
sales = df[['Product Name', 'quantity', 'Order Date', 'Sales']]
sales.columns = ['product_name', 'quantity', 'sale_date', 'revenue']
sales.to_csv('data_sales.csv', index=False, encoding='utf-8-sig')

print("Đã tách xong 3 file: data_categories.csv, data_products.csv, data_sales.csv")

# 3. Tạo file script SQL `import_data.sql`

In [ ]:
import pandas as pd

# 1. Đọc dữ liệu đã làm sạch (Sử dụng latin1 hoặc utf-8-sig tùy file của bạn)
df = pd.read_csv('cleaned_data.csv', encoding='utf-8-sig')

# Chuẩn hóa tên cột để dễ xử lý trong code
# Giả sử cột Category là 'Category', Product là 'Product Name', Sales là 'Sales', Date là 'Order Date'
df['Order Date'] = pd.to_datetime(df['Order Date']).dt.strftime('%Y-%m-%d')

with open('import_data.sql', 'w', encoding='utf-8') as f:
    # f.write("-- SQL Script Import Data\n\n")

    # --- BƯỚC 1: INSERT INTO Categories ---
    f.write("-- 1. Insert Categories\n")
    categories = df['Category'].unique()
    for cat in categories:
        # Xử lý dấu nháy đơn trong tên nếu có để tránh lỗi SQL
        cat_safe = cat.replace("'", "''")
        f.write(f"INSERT INTO Categories (category_name) VALUES ('{cat_safe}');\n")
    f.write("\n")

    # --- BƯỚC 2: INSERT INTO Products ---
    f.write("-- 2. Insert Products (Linking to Category ID)\n")
    products = df[['Product Name', 'Category', 'Sales']].drop_duplicates(subset=['Product Name'])
    for _, row in products.iterrows():
        p_name = row['Product Name'].replace("'", "''")
        cat_name = row['Category'].replace("'", "''")
        price = row['Sales'] # Tạm lấy Sales làm giá gốc nếu không có cột Price
        
        # Dùng subquery để lấy category_id dựa trên tên category
        sql = f"INSERT INTO Products (product_name, category_id, price) " \
              f"SELECT '{p_name}', category_id, {price} FROM Categories WHERE category_name = '{cat_name}';\n"
        f.write(sql)
    f.write("\n")

    # --- BƯỚC 3: INSERT INTO Sales ---
    f.write("-- 3. Insert Sales (Linking to Product ID)\n")
    for _, row in df.iterrows():
        p_name = row['Product Name'].replace("'", "''")
        qty = 1 # Mặc định số lượng là 1 như đã thống nhất
        s_date = row['Order Date']
        rev = row['Sales']
        
        # Dùng subquery để lấy product_id dựa trên tên sản phẩm
        sql = f"INSERT INTO Sales (product_id, quantity, sale_date, revenue) " \
              f"SELECT product_id, {qty}, '{s_date}', {rev} FROM Products WHERE product_name = '{p_name}';\n"
        f.write(sql)

print("Đã tạo file import_data.sql thành công!")